# MPC Noah's Ark — Seven-Layer Reasoning (V13)

Offline, synthetic, **not a real bounty scanner**. This notebook drives the actual GitHub `lib/noahs-ark-reasoning.mjs` implementation through the existing Node CLI, then compares selected candidates, missing evidence and adversarial cross-checks. The 239 MHA research hooks are separate from 24 native implemented MPC evaluators.

This notebook executes the three exact scenarios below. The Drive workbook's four manually authored planning rows are a separate historical artifact; they are not four recorded notebook executions. Each run records the actual checkout commit, dirty state, source fingerprints, and Node/Python versions. A deterministic source-derived SQLite filename preserves older caches and reuses an unchanged compatible cache; an incompatible or corrupted cache still fails admission.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json, subprocess, os, csv, hashlib, platform
ROOT = Path.cwd()
for candidate in [ROOT, *ROOT.parents]:
    if (candidate / "scripts" / "method-atlas-cli.mjs").exists():
        ROOT = candidate
        break
else:
    raise RuntimeError("Open this notebook from a checked-out Machine-Bug-Tool repository")
def command_text(args):
    return subprocess.run(args, cwd=ROOT, capture_output=True, text=True, timeout=30, check=True).stdout.rstrip("\r\n")
run_started_at = datetime.now(timezone.utc)
worktree_status = command_text(["git", "status", "--porcelain=v1", "--untracked-files=normal"])
run_context = {
    "git_commit": command_text(["git", "rev-parse", "HEAD"]),
    "git_tree": command_text(["git", "rev-parse", "HEAD^{tree}"]),
    "worktree_dirty": bool(worktree_status),
    "worktree_status": worktree_status.splitlines(),
    "node_version": command_text(["node", "--version"]),
    "python_version": platform.python_version(),
    "started_at": run_started_at.isoformat(),
    "source_identity_scope": "Actual local files at execution; a dirty worktree is not the recorded commit alone."
}
print("Repo:", ROOT)
print("Commit:", run_context["git_commit"], "| dirty:", run_context["worktree_dirty"])
print("Node:", run_context["node_version"], "| Python:", run_context["python_version"])
print("No external target traffic is performed.")


In [ ]:
files = ["candidates.json", "expansion-2026-v2.json", "expansion-evidence-intent-v3.json",
         "expansion-computation-schools-v4.json", "expansion-nasa-chip-cloud-v5.json",
         "expansion-abnormal-meta-v6.json", "expansion-optical-v8.json"]
catalogs = [json.loads((ROOT/"method-atlas"/name).read_text()) for name in files]
methods = [m for c in catalogs for m in c["methods"]]
sources = [s for c in catalogs for s in c["sources"]]
relations = json.loads((ROOT/"method-atlas"/"method-relations.json").read_text())["relationships"]
assert len(methods) == 239 and len(sources) == 65 and len(relations) == 212
ids = {m["method_id"] for m in methods}
assert len(ids) == 239 and all(x["method_id"] in ids and x["related_method_id"] in ids for x in relations)
print("Bundled-source metadata counts:", len(methods), "candidate methods /", len(sources), "source locators /", len(relations), "proposed links")

def canonical_bytes(value):
    return json.dumps(value, sort_keys=True, separators=(",", ":"), ensure_ascii=False).encode("utf-8")
def fingerprint_paths(paths):
    file_hashes = {path: hashlib.sha256((ROOT/path).read_bytes()).hexdigest() for path in sorted(paths)}
    return {"sha256": hashlib.sha256(canonical_bytes(file_hashes)).hexdigest(), "files": file_hashes}
def current_cache_inputs():
    source_paths = ["scripts/method-atlas-cli.mjs"] + [
        p.relative_to(ROOT).as_posix() for p in (ROOT/"lib").rglob("*")
        if p.is_file() and p.suffix in {".mjs", ".json"}
    ]
    return {
        "source": fingerprint_paths(source_paths),
        "schema": fingerprint_paths(["method-atlas/schema.sql"]),
        "catalog": fingerprint_paths(["method-atlas/"+name for name in files]+["method-atlas/method-relations.json"]),
        "taxonomy": fingerprint_paths(["method-atlas/reclassification-policy-v7.json", "lib/method-reclassification.mjs"])
    }
cache_inputs = current_cache_inputs()
cache_identity = {"version": "MPC_NOAHS_ARK_NOTEBOOK_CACHE_V1",
                  "input_sha256": {name: item["sha256"] for name, item in cache_inputs.items()},
                  "node_version": run_context["node_version"]}
cache_fingerprint = hashlib.sha256(canonical_bytes(cache_identity)).hexdigest()
cache_path = ROOT/".sites-runtime"/("noahs-ark-v13-notebook-"+cache_fingerprint+".sqlite")
run_context.update({"cache_path": str(cache_path), "cache_fingerprint": cache_fingerprint,
                    "cache_identity": cache_identity, "cache_inputs": cache_inputs,
                    "cache_existed_at_start": cache_path.exists()})
print("Source-derived cache:", cache_path)


In [ ]:
scenarios = [
 {"name":"Synthetic ledger finality","atom":{"subject_id":"fixture:transaction","atom_id":"fixture:ledger","dimensions":["MONEY","FINALITY"],"source_refs":["fixture:owned-ledger"],"external_source_refs":[]},
  "method_receipts":[{"method_id":i,"input_state":"AVAILABLE","negative_control_state":"AVAILABLE","falsifier_state":"AVAILABLE","estimated_cost_units":cost}
                     for i,cost in [("MHA-0052",1),("MHA-0053",2),("MHA-0036",3)]]},
 {"name":"Cloud configuration — unknown inputs","atom":{"subject_id":"fixture:cloud","atom_id":"fixture:policy","dimensions":["CLOUD","AUTHORITY"],"source_refs":["fixture:config-snapshot"],"external_source_refs":[]},"method_receipts":[]},
 {"name":"Casino meter — source unavailable","atom":{"subject_id":"fixture:game","atom_id":"fixture:meter","dimensions":["MONEY","STATE"],"source_refs":[],"external_source_refs":[]},"method_receipts":[]}
]
env = os.environ.copy()
env["MPC_METHOD_ATLAS_DB"] = str(cache_path)
def cli(command, payload=None):
    args = ["node", str(ROOT/"scripts"/"method-atlas-cli.mjs"), command]
    if payload is not None: args.append(json.dumps(payload))
    p = subprocess.run(args, cwd=ROOT, env=env, capture_output=True, text=True, timeout=120, check=True)
    return json.loads(p.stdout)
init = cli("init")
assert init["methods"] == 239
run_context["admitted_atlas_fingerprints"] = {key: init[key] for key in
    ["seed_fingerprint", "schema_fingerprint", "content_fingerprint", "taxonomy_policy_sha256", "taxonomy_compiler_sha256"]}
results = [(s["name"], cli("reason", {k:v for k,v in s.items() if k!="name"})) for s in scenarios]
assert cache_inputs == current_cache_inputs(), "Local cache inputs changed during execution; retain receipts and rerun the exact source."
run_context["cache_inputs_stable_through_execution"] = True
for name, receipt in results:
    outcome=receipt["outcome_vector"]
    print(name, "|", receipt["status"], "| covered:", len(outcome["covered_dimensions"]),
          "| selected:", outcome["selected_count"], "| paired:", outcome["paired_challengers"])


In [ ]:
rows=[]
for name,receipt in results:
    v=receipt["outcome_vector"]
    rows.append({"scenario":name,"status":receipt["status"],"considered":v["catalog_methods_considered"],
      "triggered":v["structurally_triggered"],"eligible":v["structurally_eligible"],
      "selected":v["selected_count"],"pairs":v["paired_challengers"],
      "dimensions_covered":len(v["covered_dimensions"]),"dimensions_unresolved":len(v["unresolved_dimensions"]),
      "controls_ready":v["negative_controls_declared_ready"],"falsifiers_ready":v["falsifiers_declared_ready"],
      "declared_cost_units":v["declared_total_cost_units"],
      "source_authenticated":False,"method_executed":False,"security_impact_verified":False,
      "git_commit":run_context["git_commit"],"worktree_dirty":run_context["worktree_dirty"],
      "node_version":run_context["node_version"],"python_version":run_context["python_version"],
      "cache_fingerprint":cache_fingerprint})
run_context["completed_at"] = datetime.now(timezone.utc).isoformat()
run_stem = "noahs-ark-v13-"+cache_fingerprint+"-"+run_started_at.strftime("%Y%m%dT%H%M%S%fZ")
out = ROOT/".sites-runtime"/(run_stem+"-synthetic-outcomes.csv")
out.parent.mkdir(parents=True,exist_ok=True)
with out.open("x",newline="") as f:
    w=csv.DictWriter(f,fieldnames=list(rows[0]));w.writeheader();w.writerows(rows)
receipt_path = out.with_suffix(".json")
with receipt_path.open("x") as f:
    json.dump({"run_context":run_context,"synthetic_inputs":scenarios,"atlas_init":init,
               "scenario_results":results,"summary_rows":rows},f,ensure_ascii=False,indent=2)
    f.write("\n")
print("CSV:", out)
print("Execution receipt:", receipt_path)
for row in rows: print(row)


In [ ]:
for name,r in results:
    assert r["no_method_executed"] and r["no_source_authentication"] and not r["canonical_promotion"]
    assert len(r["method_consideration"]) == len(methods)
    assert len(r["reasoning_layers"]) == 7
    assert all(not p["challenger_executed"] and p["native_evidence_independence"]=="NOT_ESTABLISHED" for p in r["proposed_pairs"])
print("All 239 candidates were accounted for in each offline scenario; no method execution or security finding claimed.")

## Interpretation

Select the method with ready inputs, useful negative controls and falsifier, and coverage of unresolved typed dimensions. Pair it with a challenger where there is a declared method link. **Never use a method name, model agreement, source locator, or payoff hypothesis as authentication or proof.**

The three inputs above are preserved synthetic fixtures. This notebook executes those three fixtures only; it does not supply a fourth model or infer missing readiness for the workbook's `Meta method graph` planning row.

The workbook at native ID `1hsuWvDGTMJE7_UFJaqlw8hzofDRl-9h2ZtphMDFh9Bw`, revision `7` read on 2026-10-09, contains four manually authored planning rows. Its cloud row labels the decision `BLOCKED_INPUT_SCHEMA` and names `MHA-0170` / `MHA-0095`. The actual V13 baseline notebook cloud fixture returned `BLOCKED_NO_READY_METHOD`, with no selected method or paired challenger. Keep that authored label and those proposed candidates as historical planning data; use each execution receipt for the actual result. This notebook makes no workbook changes or claim of automatic synchronization.

The source-derived SQLite path changes when its recorded source, schema, catalog, taxonomy, or Node identity changes. It does not delete, migrate, or rename the older `noahs-ark-v13-notebook.sqlite`; the CLI still rejects an incompatible or corrupted existing cache. Identical inputs reuse the same cache. Timestamped output receipts preserve earlier runs and record the actual Git commit, dirty state, source fingerprints, and runtimes.

Proposed production progression: native-source readback → typed atoms → structural method shortlist → separately authorized finite evaluator → opposing controls → exact PASS checkpoint. The notebook does not sync with Google Drive or GitHub; the Drive workbook is a separate connected artifact.
